# High-Performance Pandas met `eval()` en `query()`

Deze notebook biedt een uitgebreide gids voor het gebruik van `eval()` en `query()` in pandas. Deze tools helpen bij het schrijven van efficiënte en leesbare code, vooral bij grote datasets.

## 1. `query()` en `eval()`

### Waarom deze functies?

Bij grote DataFrames kunnen complexe expressies zoals filters en berekeningen leiden tot code die moeilijk te lezen en traag is. Functies zoals `query()` en `eval()` bieden een efficiënte en compacte manier om dergelijke operaties uit te voeren.  

**Voorbeeld:**

In [8]:
import pandas as pd
import numpy as np
import time

data = pd.DataFrame({
    'A': np.random.rand(1_000_000),  # willekeurige getallen tussen 0 en 1
    'B': np.random.rand(1_000_000),
    'C': np.random.randint(0, 100, 1_000_000) # 1 miljoen ints tssn 0 & 100
})


data.head()

,A,B,C
0,0.485816,0.776197,24
1,0.631161,0.838987,1
2,0.831760,0.990258,27
3,0.347460,0.814997,50
4,0.589570,0.113178,2


In [13]:
# Zonder query notatie
start_time = time.time()
gemiddelde_A_B = data[(data['A'] > 0.5) & (data['B'] < 0.5)]
end_time = time.time()

print(f"Execution time: {end_time - start_time:.6f} seconds")

Execution time: 0.005492 seconds


In [14]:
# Met query notatie
start_time = time.time()
gemiddelde_A_B_query = data.query('A > 0.5 and B < 0.5')
end_time = time.time()

print(f"Execution time: {end_time - start_time:.6f} seconds")

Execution time: 0.013634 seconds


De `query()`-versie is eenvoudiger te lezen.

## 2. Gebruik van `pandas.eval()` voor Efficiënte Operaties

### Wat doet `eval()`?

`eval()` voert een stringexpressie uit als een efficiënte operatie binnen pandas, waarbij berekeningen direct worden uitgevoerd zonder het maken van tussenliggende objecten in het geheugen. Dit maakt het bijzonder geschikt voor het uitvoeren van complexe expressies op grote datasets.

**Hoe gebruik je `eval()`?

1. Formuleer een stringexpressie die kolomnamen en operaties bevat.
2. Roep `pd.eval()` aan voor globale berekeningen of `DataFrame.eval()` voor berekeningen binnen een specifieke DataFrame.

**Welke expressies kun je invoeren?**

- **Rekenkundige operaties**: Gebruik `+`, `-`, `*`, `/`, `**` (macht), en `%` (modulo).
- **Logische operaties**: Gebruik `and`, `or`, en `not` voor booleaanse vergelijkingen.
- **Vergelijkingen**: Gebruik `>`, `<`, `>=`, `<=`, `==`, en `!=`.
- **Combinaties**: Combineer rekenkundige en logische operaties in één string.

**Voorbeelden:**

In [15]:
# Optelling van kolommen
data['Som'] = pd.eval('data.A + data.B')

# Logische filtering
data['Filter'] = pd.eval('(data.A > 0.5) & (data.B < 0.5)')

# Geavanceerde berekeningen
resultaat = pd.eval('data.A ** 2 + 3 * data.B - data.C / 2')

`eval()` maakt de code compacter, sneller en helpt geheugenruimte te besparen door het vermijden van tijdelijke objecten.

**Voorbeeld:**

In [16]:
# Kolommen optellen zonder eval
resultaat = data['A'] + data['B']

# Kolommen optellen met eval
resultaat_eval = pd.eval('data.A + data.B')

**Voordelen:**

- Sneller bij grote datasets.
- Minder geheugengebruik.

## 3. **Gebruik van `DataFrame.eval()` voor Kolom-Operaties**

### Wat is `DataFrame.eval()`?

`DataFrame.eval()` stelt je in staat om kolom-berekeningen direct binnen een DataFrame uit te voeren.

**Voorbeeld:**

In [9]:
# Nieuwe kolom toevoegen zonder eval
data['D'] = data['A'] * data['B']

# Nieuwe kolom toevoegen met eval
data.eval('D = A * B', inplace=True)

Hierdoor wordt de code compacter en efficiënter.

## 4. **Gebruik van `DataFrame.query()` voor Filtering**

### Wat is `DataFrame.query()`?

`DataFrame.query()` is een krachtige methode waarmee je filtering kunt uitvoeren op een DataFrame met behulp van stringexpressies. Het biedt een leesbaarder alternatief voor traditionele filtermethoden door voorwaarden op te geven in een SQL-achtige syntax.

### Hoe werkt `query()`?

- Het werkt door een string met logische en vergelijkingsuitdrukkingen te evalueren.
- Je kunt direct verwijzen naar kolomnamen zonder aanhalingstekens.
- Voor kolomnamen met spaties of speciale tekens gebruik je backticks (` `).

### Welke expressies kun je invoeren?

- **Vergelijkingen**: `<`, `>`, `<=`, `>=`, `==`, `!=`.
- **Logische operatoren**: `and` (voor `&`), `or` (voor `|`), `not` (voor `~`).
- **Combinaties**: Combineer meerdere vergelijkingen met haakjes voor duidelijkheid.

### Voorbeelden van gebruik:

In [28]:
# Filter op waarden in kolom A en kolom B
geselecteerd = data.query('A > 0.5 and B < 0.5')

# Gebruik van backticks voor speciale kolomnamen
speciale_data = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6], 'C': [7, 8, 9], 'D': [10, 11, 12]})
print(f"Speciale data:\n\n{speciale_data}")

resultaat = speciale_data.query('`A` < 2 or `D` > 11')
print(f"Speciale data na query `A` < 1 or `D` > 10:\n\n{resultaat}")

# Dynamische filtering met variabelen
waarde = 50
resultaat = data.query('C > @waarde')

Speciale data:

   A  B  C   D
0  1  4  7  10
1  2  5  8  11
2  3  6  9  12
Speciale data na query `A` < 1 or `D` > 10:

   A  B  C   D
0  1  4  7  10
2  3  6  9  12


## 5. **Prestatievergelijking: Wanneer Gebruik je Deze Functies?**

### Benchmark: `eval()` en `query()` vs. traditionele methoden

Test de prestaties van beide methoden met een grote dataset:

In [29]:
import time

# Dataset
n = 10**8
data = pd.DataFrame({
    'A': np.random.rand(n),
    'B': np.random.rand(n),
    'C': np.random.randint(0, 100, n)
})

# Traditionele methode
start = time.time()
resultaat = data[(data['A'] > 0.5) & (data['B'] < 0.5)]
eind = time.time()
print(f"Traditionele methode: {eind - start:.4f} seconden")

# Met query
start = time.time()
resultaat_query = data.query('A > 0.5 and B < 0.5')
eind = time.time()
print(f"Met query: {eind - start:.4f} seconden")

Traditionele methode: 0.6635 seconden
Met query: 2.3465 seconden


### Richtlijnen:

1. Gebruik `query()` voor complexe filteroperaties.
2. Gebruik `eval()` voor berekeningen tussen kolommen.
3. Kies traditionele methoden voor kleine datasets of simpele operaties.

In deze specifieke benchmark is de traditionele methode sneller omdat deze direct gebruikmaakt van geoptimaliseerde pandas-logica zonder de overhead van string parsing en evaluatie. Voor kritieke prestaties wordt meestal aanbevolen om de traditionele aanpak te gebruiken, tenzij leesbaarheid en dynamiek belangrijker zijn.